# Notebook 1: HCC/RAF Scoring — Deep Dive
## Medicare Risk Adjustment Factor (RAF) Methodology

This notebook walks through the CMS HCC v28 model step-by-step:
- ICD-10 → HCC mapping logic
- Demographic + disease + interaction coefficients
- RAF score calculation for individual beneficiaries
- Cohort-level RAF distribution analysis
- PMPM cost estimation from RAF scores

**Reference:** CMS Medicare Advantage Risk Adjustment Model, CY2024 (v28)


In [4]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import display

from medicare_raf.modeling.hcc_mapper import (
    map_icd10_to_hcc, get_hcc_coefficient, get_hcc_label,
    HCC_COEFFICIENTS_V28, ICD10_TO_HCC
)
from medicare_raf.modeling.raf_calculator import (
    calculate_raf, calculate_raf_batch,
    estimate_pmpm_cost, summarise_cohort_raf
)
from medicare_raf.data.data_generator import generate_beneficiary_cohort

sns.set_style("whitegrid")
ACCENT = "#1B4F72"
print("Imports OK")


Imports OK


## 1. Understanding HCC Categories

The CMS v28 model maps ICD-10 diagnosis codes to Hierarchical Condition Categories (HCCs). Each HCC has a coefficient representing the additional expected cost above the demographic baseline.

In [ ]:
# Show all HCC coefficients sorted by risk weight
hcc_df = pd.DataFrame([
    {"HCC": hcc, "Label": get_hcc_label(hcc), "Coefficient": coeff}
    for hcc, coeff in sorted(HCC_COEFFICIENTS_V28.items(), key=lambda x: -x[1])
])

print(f"Total HCC categories in this model: {len(hcc_df)}")
print(f"\nHighest risk HCCs:")
display(hcc_df.head(10).to_string(index=False))
print(f"\nLowest risk HCCs:")
display(hcc_df.tail(5).to_string(index=False))


## 2. Single Beneficiary RAF Calculation

Let's walk through the RAF calculation for a real clinical scenario: a 76-year-old female with CHF, atrial fibrillation, and Type 2 diabetes with CKD.

In [ ]:
# Clinical scenario: complex Medicare patient
example = {
    "age": 76,
    "sex": "F",
    "conditions": "CHF + Atrial Fibrillation + T2DM with CKD Stage 4",
    "icd10_codes": ["I500", "I5022", "I481", "E1141", "N184"]
}

result = calculate_raf(
    age=example["age"],
    sex=example["sex"],
    icd10_codes=example["icd10_codes"]
)

print(f"Patient: {example['age']}yo {example['sex']} — {example['conditions']}")
print(f"{'─'*55}")
print(f"  Demographic RAF:     {result['demographic_raf']:.4f}")
print(f"  HCC RAF:             {result['hcc_raf']:.4f}")
print(f"  Interaction RAF:     {result['interaction_raf']:.4f}  ← CHF×AFib bonus")
print(f"  {'─'*45}")
print(f"  TOTAL RAF SCORE:     {result['raf_score']:.4f}")
print(f"  Estimated annual cost: ${estimate_pmpm_cost(result['raf_score']):,.0f}")
print(f"\n  HCCs triggered: {result['hcc_labels']}")
print(f"\n  HCC detail:")
for hcc, coeff in result['hcc_details'].items():
    print(f"    HCC {hcc:3d}  {get_hcc_label(hcc):<40}  +{coeff:.3f}")


## 3. Comparing RAF Across Patient Profiles

How does RAF vary across common Medicare patient archetypes?

In [ ]:
profiles = [
    {"name": "Healthy 67yo Male",     "age": 67, "sex": "M", "codes": []},
    {"name": "T2DM 72yo Female",      "age": 72, "sex": "F", "codes": ["E119"]},
    {"name": "CHF + AFib 78yo Male",  "age": 78, "sex": "M", "codes": ["I500", "I481"]},
    {"name": "COPD + T2DM 74yo F",    "age": 74, "sex": "F", "codes": ["J441", "E1141"]},
    {"name": "CKD Stage 5 80yo M",    "age": 80, "sex": "M", "codes": ["N185", "E1141"]},
    {"name": "Metastatic Cancer 71yo F", "age": 71, "sex": "F", "codes": ["C7800", "C800"]},
    {"name": "ESRD + CHF + T2DM 75yo M","age": 75, "sex": "M", "codes": ["N186","I500","E1141"]},
]

rows = []
for p in profiles:
    r = calculate_raf(p["age"], p["sex"], p["codes"])
    rows.append({
        "Profile": p["name"],
        "RAF Score": r["raf_score"],
        "Est. Annual Cost": f"${estimate_pmpm_cost(r['raf_score']):,.0f}",
        "HCC Count": len(r["hccs"]),
    })

prof_df = pd.DataFrame(rows)
display(prof_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
colors = [ACCENT if r > 2 else "#2E86C1" if r > 1 else "#85C1E9"
          for r in prof_df["RAF Score"]]
bars = ax.barh(prof_df["Profile"], prof_df["RAF Score"], color=colors, edgecolor="white")
ax.axvline(1.0, color="red", linestyle="--", linewidth=1.5, alpha=0.7, label="Average (RAF=1.0)")
for bar, val in zip(bars, prof_df["RAF Score"]):
    ax.text(bar.get_width() + 0.03, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}", va="center", fontsize=9)
ax.set_xlabel("RAF Score", fontsize=11)
ax.set_title("RAF Scores by Patient Profile", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Cohort-Level RAF Distribution (50,000 Beneficiaries)

In [ ]:
print("Generating 50,000-beneficiary cohort (this takes ~60 seconds)...")
cohort = generate_beneficiary_cohort(n=50_000)
cohort_raf = calculate_raf_batch(cohort)

summary = summarise_cohort_raf(cohort_raf)
print(f"\nCohort RAF Summary:")
for k, v in summary.items():
    print(f"  {k:<30} {v:>15,}" if isinstance(v, (int, float)) else f"  {k}: {v}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution
axes[0].hist(cohort_raf["raf_score"], bins=80, color=ACCENT, alpha=0.8, edgecolor="white")
axes[0].axvline(1.0, color="red", linestyle="--", linewidth=1.5, label="Average")
axes[0].axvline(cohort_raf["raf_score"].mean(), color="orange", linestyle="--",
                linewidth=1.5, label=f"Mean={cohort_raf['raf_score'].mean():.2f}")
axes[0].set_xlabel("RAF Score"); axes[0].set_ylabel("Count")
axes[0].set_title("RAF Distribution (N=50,000)"); axes[0].legend(fontsize=8)

# By risk tier
tier_order = ["low", "moderate", "high"]
raf_by_tier = cohort_raf.groupby("risk_tier")["raf_score"].mean().reindex(tier_order)
bars = axes[1].bar(tier_order, raf_by_tier.values,
                   color=["#85C1E9","#2E86C1", ACCENT], edgecolor="white")
for bar, val in zip(bars, raf_by_tier.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{val:.2f}", ha="center", fontsize=10)
axes[1].set_xlabel("Risk Tier"); axes[1].set_ylabel("Mean RAF")
axes[1].set_title("Mean RAF by Risk Tier")

# Cost distribution
costs = cohort_raf["raf_score"].apply(estimate_pmpm_cost)
axes[2].hist(costs, bins=80, color="#2E86C1", alpha=0.8, edgecolor="white")
axes[2].axvline(costs.mean(), color="orange", linestyle="--",
                label=f"Mean=${costs.mean():,.0f}")
axes[2].set_xlabel("Estimated Annual Cost ($)"); axes[2].set_ylabel("Count")
axes[2].set_title("Estimated Annual Cost Distribution")
axes[2].legend(fontsize=8)
axes[2].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}K"))

plt.suptitle("CMS HCC v28 RAF Analysis — 50,000 Medicare Beneficiaries",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
print(f"\nTotal estimated portfolio cost: ${costs.sum():,.0f}")


## 5. HCC Prevalence in the Cohort

In [ ]:
# Count HCC prevalence across cohort
from collections import Counter
from src.hcc_mapper import get_hcc_label

all_hccs = []
for hcc_list in cohort_raf["hccs"]:
    if isinstance(hcc_list, list):
        all_hccs.extend(hcc_list)

hcc_counts = Counter(all_hccs)
hcc_prev = pd.DataFrame([
    {"HCC": hcc, "Label": get_hcc_label(hcc),
     "Count": cnt, "Prevalence %": round(cnt/len(cohort_raf)*100, 1),
     "Coefficient": HCC_COEFFICIENTS_V28.get(hcc, 0)}
    for hcc, cnt in sorted(hcc_counts.items(), key=lambda x: -x[1])
]).head(15)

display(hcc_prev.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(hcc_prev["Label"][::-1], hcc_prev["Prevalence %"][::-1],
        color=ACCENT, alpha=0.85, edgecolor="white")
ax.set_xlabel("Prevalence (%)", fontsize=11)
ax.set_title("Top 15 HCC Prevalence — 50,000 Medicare Beneficiaries",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
